In [13]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
import os

PROCESSED_PATH = '/content/drive/MyDrive/text_mining_ds2/processed/'
MODELS_PATH    = '/content/drive/MyDrive/text_mining_ds2/models/'

print("Fichiers processed :")
for f in os.listdir(PROCESSED_PATH):
    print(f)

print("\nFichiers models :")
for f in os.listdir(MODELS_PATH):
    print(f)

Fichiers processed :
ds2_eda.csv
tfidf_vectorizer_ds2.pkl
y_test_ds2.npy
X_train_pad_ds2.npy
X_val_pad_ds2.npy
ds2_preprocessed.csv
y_train_ds2.npy
keras_tokenizer_ds2.pkl
y_val_ds2.npy
X_test_pad_ds2.npy
ds2_params.json

Fichiers models :
lstm_model.py
__pycache__


In [16]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import json
import os
import time
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
import sys
sys.path.append(MODELS_PATH)
from lstm_model import build_lstm_model

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from tensorflow.keras.callbacks import EarlyStopping

print("Imports OK")
print("GPU dispo :", len(tf.config.list_physical_devices('GPU')) > 0)

Imports OK
GPU dispo : True


## Bloc 0 — Imports & Chargement des données

In [17]:
with open(PROCESSED_PATH + 'ds2_params.json') as f:
    params = json.load(f)

VOCAB_SIZE  = params['vocab_size']
MAXLEN      = params['maxlen']
NUM_CLASSES = params['num_classes']

X_train = np.load(PROCESSED_PATH + 'X_train_pad_ds2.npy')
X_val   = np.load(PROCESSED_PATH + 'X_val_pad_ds2.npy')
X_test  = np.load(PROCESSED_PATH + 'X_test_pad_ds2.npy')
y_train = np.load(PROCESSED_PATH + 'y_train_ds2.npy')
y_val   = np.load(PROCESSED_PATH + 'y_val_ds2.npy')
y_test  = np.load(PROCESSED_PATH + 'y_test_ds2.npy')

with open(PROCESSED_PATH + 'tfidf_vectorizer_ds2.pkl', 'rb') as f:
    tfidf = pickle.load(f)

with open(PROCESSED_PATH + 'keras_tokenizer_ds2.pkl', 'rb') as f:
    tokenizer = pickle.load(f)

df = pd.read_csv(PROCESSED_PATH + 'ds2_preprocessed.csv')

print(f"X_train : {X_train.shape}")
print(f"VOCAB_SIZE : {VOCAB_SIZE} | MAXLEN : {MAXLEN} | NUM_CLASSES : {NUM_CLASSES}")

X_train : (6035, 24)
VOCAB_SIZE : 10000 | MAXLEN : 24 | NUM_CLASSES : 2


## Bloc 1 — Calcul des scores MI (base pour le GA)

In [18]:
from sklearn.feature_selection import mutual_info_classif

df_train = df.iloc[:len(y_train)]
X_tfidf_train = tfidf.transform(df_train['text_clean'])

print("Calcul des scores MI...")
mi_scores = mutual_info_classif(X_tfidf_train, y_train, random_state=42)

vocab = tfidf.get_feature_names_out()
mi_df = pd.DataFrame({'mot': vocab, 'mi_score': mi_scores})
mi_df = mi_df.sort_values('mi_score', ascending=False).reset_index(drop=True)

print("Top 10 mots les plus informatifs :")
print(mi_df.head(10).to_string(index=False))

Calcul des scores MI...
Top 10 mots les plus informatifs :
   mot  mi_score
  الله  0.069911
    لا  0.052863
    ما  0.052654
 اللهم  0.030759
  اللي  0.026897
 والله  0.020285
 الخير  0.018794
  يارب  0.017953
   يوم  0.016772
الهلال  0.015937


## Bloc 2 — Fonction fitness (LSTM)



In [19]:
TOP_K_SEARCH = 1000
search_space = mi_df.head(TOP_K_SEARCH)['mot'].tolist()

def evaluate_individual(individual):
    selected = [search_space[i] for i, bit in enumerate(individual) if bit == 1]

    if len(selected) < 10:
        return (0.0,)

    new_vocab = {word: idx+1 for idx, word in enumerate(selected)}
    old_to_new = {}
    for word, new_idx in new_vocab.items():
        old_idx = tokenizer.word_index.get(word)
        if old_idx is not None:
            old_to_new[old_idx] = new_idx

    def remap(X):
        X_new = np.zeros_like(X)
        for old_idx, new_idx in old_to_new.items():
            X_new[X == old_idx] = new_idx
        return X_new

    Xtr = remap(X_train)
    Xvl = remap(X_val)
    vocab_size = len(old_to_new) + 1

    model = build_lstm_model(
        vocab_size=vocab_size,
        maxlen=MAXLEN,
        n_classes=NUM_CLASSES
    )

    es = EarlyStopping(monitor='val_loss', patience=2,
                       restore_best_weights=True, verbose=0)

    model.fit(
        Xtr, y_train,
        validation_data=(Xvl, y_val),
        epochs=5,
        batch_size=64,
        callbacks=[es],
        verbose=0
    )

    y_pred = np.argmax(model.predict(Xvl, verbose=0), axis=1)
    f1 = f1_score(y_val, y_pred, average='macro')
    return (f1,)

print("Fitness LSTM définie")
print("Chaque évaluation ≈ 15s sur GPU")


Fitness LSTM définie
Chaque évaluation ≈ 15s sur GPU


## Bloc 3 — Configuration DEAP

In [20]:
from deap import base, creator, tools
import random

if 'FitnessMax' not in dir(creator):
    creator.create('FitnessMax', base.Fitness, weights=(1.0,))
if 'Individual' not in dir(creator):
    creator.create('Individual', list, fitness=creator.FitnessMax)

toolbox = base.Toolbox()
toolbox.register('attr_bool', np.random.randint, 0, 2)
toolbox.register('individual', tools.initRepeat,
                 creator.Individual, toolbox.attr_bool, n=TOP_K_SEARCH)
toolbox.register('population', tools.initRepeat, list, toolbox.individual)
toolbox.register('evaluate', evaluate_individual)
toolbox.register('mate',     tools.cxTwoPoint)
toolbox.register('mutate',   tools.mutFlipBit, indpb=0.01)
toolbox.register('select',   tools.selTournament, tournsize=3)

print("DEAP configuré")
print(f"Taille chromosome : {TOP_K_SEARCH} bits")

DEAP configuré
Taille chromosome : 1000 bits


## Bloc 4 — Lancement GA

In [22]:
POP_SIZE = 10
N_GEN    = 5
CXPB     = 0.7
MUTPB    = 0.01

population = toolbox.population(n=POP_SIZE)
hof        = tools.HallOfFame(1)
logbook    = tools.Logbook()
logbook.header = ['gen', 'nevals', 'max', 'mean']

stats = tools.Statistics(lambda ind: ind.fitness.values)
stats.register('max',  np.max)
stats.register('mean', np.mean)

print(f"Démarrage GA : {POP_SIZE} individus x {N_GEN} générations")
print("=" * 50)

start = time.time()

fitnesses = list(map(toolbox.evaluate, population))
for ind, fit in zip(population, fitnesses):
    ind.fitness.values = fit

for gen in range(N_GEN):
    offspring = toolbox.select(population, len(population))
    offspring = list(map(toolbox.clone, offspring))

    for c1, c2 in zip(offspring[::2], offspring[1::2]):
        if np.random.random() < CXPB:
            toolbox.mate(c1, c2)
            del c1.fitness.values
            del c2.fitness.values

    for mutant in offspring:
        if np.random.random() < MUTPB:
            toolbox.mutate(mutant)
            del mutant.fitness.values

    invalid = [ind for ind in offspring if not ind.fitness.valid]
    fitnesses = list(map(toolbox.evaluate, invalid))
    for ind, fit in zip(invalid, fitnesses):
        ind.fitness.values = fit

    population[:] = offspring
    hof.update(population)

    record = stats.compile(population)
    logbook.record(gen=gen, nevals=len(invalid), **record)

    elapsed = (time.time() - start) / 60
    print(f"Gen {gen+1:02d}/{N_GEN} | Max: {record['max']:.4f} | Mean: {record['mean']:.4f} | Temps: {elapsed:.1f} min")

print(f"\nGA terminé en {elapsed:.1f} minutes")
print(f"Meilleur fitness : {hof[0].fitness.values[0]:.4f}")
print(f"Features sélectionnées : {sum(hof[0])}/{TOP_K_SEARCH}")

Démarrage GA : 10 individus x 5 générations
Gen 01/5 | Max: 0.5947 | Mean: 0.4503 | Temps: 16.3 min
Gen 02/5 | Max: 0.5947 | Mean: 0.4488 | Temps: 19.9 min
Gen 03/5 | Max: 0.5781 | Mean: 0.4722 | Temps: 26.8 min
Gen 04/5 | Max: 0.5815 | Mean: 0.5443 | Temps: 32.2 min
Gen 05/5 | Max: 0.5931 | Mean: 0.4617 | Temps: 38.6 min

GA terminé en 38.6 minutes
Meilleur fitness : 0.5947
Features sélectionnées : 506/1000


## Bloc 5 — Évaluation finale LSTM sur features GA

In [23]:
best_features = [search_space[i] for i, bit in enumerate(hof[0]) if bit == 1]
print(f"Features sélectionnées : {len(best_features)}")

new_vocab = {word: idx+1 for idx, word in enumerate(best_features)}
old_to_new = {}
for word, new_idx in new_vocab.items():
    old_idx = tokenizer.word_index.get(word)
    if old_idx is not None:
        old_to_new[old_idx] = new_idx

def remap(X):
    X_new = np.zeros_like(X)
    for old_idx, new_idx in old_to_new.items():
        X_new[X == old_idx] = new_idx
    return X_new

X_tr_ga = remap(X_train)
X_vl_ga = remap(X_val)
X_ts_ga = remap(X_test)
new_vocab_size = len(old_to_new) + 1

# LSTM sur features GA
model_ga = build_lstm_model(
    vocab_size=new_vocab_size,
    maxlen=MAXLEN,
    n_classes=NUM_CLASSES
)

es = EarlyStopping(monitor='val_loss', patience=3,
                   restore_best_weights=True, verbose=1)

t0 = time.time()
history_ga = model_ga.fit(
    X_tr_ga, y_train,
    validation_data=(X_vl_ga, y_val),
    epochs=20,
    batch_size=64,
    callbacks=[es],
    verbose=1
)
elapsed_ga = round(time.time() - t0, 1)

y_pred_ga = np.argmax(model_ga.predict(X_ts_ga, verbose=0), axis=1)

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
results_ga = {
    'method'   : 'GA+LSTM',
    'features' : len(best_features),
    'accuracy' : round(accuracy_score(y_test, y_pred_ga), 4),
    'f1_macro' : round(f1_score(y_test, y_pred_ga, average='macro'), 4),
    'precision': round(precision_score(y_test, y_pred_ga, average='macro'), 4),
    'recall'   : round(recall_score(y_test, y_pred_ga, average='macro'), 4),
    'time_s'   : elapsed_ga
}

for k, v in results_ga.items():
    print(f"{k:<12} : {v}")

Features sélectionnées : 506
Epoch 1/20
95/95 ━━━━━━━━━━━━━━━━━━━━ 14s 104ms/step - accuracy: 0.5092 - loss: 0.6937 - val_accuracy: 0.4927 - val_loss: 0.6933
Epoch 2/20
95/95 ━━━━━━━━━━━━━━━━━━━━ 10s 105ms/step - accuracy: 0.5072 - loss: 0.6931 - val_accuracy: 0.4927 - val_loss: 0.6932
Epoch 3/20
95/95 ━━━━━━━━━━━━━━━━━━━━ 10s 102ms/step - accuracy: 0.4983 - loss: 0.6935 - val_accuracy: 0.5089 - val_loss: 0.6925
Epoch 4/20
95/95 ━━━━━━━━━━━━━━━━━━━━ 9s 92ms/step - accuracy: 0.5198 - loss: 0.6917 - val_accuracy: 0.4927 - val_loss: 0.6903
Epoch 5/20
95/95 ━━━━━━━━━━━━━━━━━━━━ 10s 108ms/step - accuracy: 0.5659 - loss: 0.6769 - val_accuracy: 0.5978 - val_loss: 0.6659
Epoch 6/20
95/95 ━━━━━━━━━━━━━━━━━━━━ 10s 108ms/step - accuracy: 0.6152 - loss: 0.6582 - val_accuracy: 0.5932 - val_loss: 0.6734
Epoch 7/20
95/95 ━━━━━━━━━━━━━━━━━━━━ 8s 89ms/step - accuracy: 0.6275 - loss: 0.6460 - val_accuracy: 0.5932 - val_loss: 0.6681
Epoch 8/20
95/95 ━━━━━━━━━━━━━━━━━━━━ 10s 107ms/step - accuracy: 0.6394 